# 크리깅 기반 범퍼 SEA 서로게이트 및 강건 최적화

`PDD_bumper_SEA.ipynb`에서 PDD로 민감도 분석 후 결정한 `DESIGN_IDX`/`NOISE_IDX`, 스케일링된 `X_scaled`/`Y`를 이어받아 진행하는 노트북(GUIDELINE.md 파이프라인 5~11단계 중 7단계부터).

지도교수 지침: 범퍼 충돌 응답은 비선형·불연속적이라 PDD 같은 전역 다항식 서로게이트보다 크리깅(Gaussian Process)이 적합. 왜 그런지 실증 비교는 `PDD_vs_Kriging_benchmark.ipynb` 참고.

구성:
1. 데이터 로딩 (DOE 생성 유틸 포함)
2. 크리깅 서로게이트 구축 및 검증
3. 강건 목적함수
4. 최적화
5. 검증

In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.model_selection import train_test_split
from scipy.stats import qmc
from scipy.optimize import minimize


def theoretical_scale(x, domain_min, domain_max, target_min=-1, target_max=1):
    # PDD_Legendre_ver4.ipynb와 동일. 다른 정의역의 변수를 공통 스케일로 맞춰 크리깅 커널 하이퍼파라미터가
    # 특정 변수에 치우치지 않게 함.
    x_std = (x - domain_min) / (domain_max - domain_min)
    scaled_x = x_std * (target_max - target_min) + target_min
    return scaled_x


def lhs_design(domain_min, domain_max, n_samples, seed=0):
    # Latin Hypercube Sampling으로 DOE 점 생성. 카티아/아바쿠스에 넘길 실제 물리 단위로 반환.
    # domain_min, domain_max: (dim, 1) 배열
    dim = domain_min.shape[0]
    sampler = qmc.LatinHypercube(d=dim, seed=seed)
    unit_samples = sampler.random(n=n_samples)  # (n_samples, dim), [0,1) 구간

    lo = domain_min.ravel()
    hi = domain_max.ravel()
    physical = lo + unit_samples * (hi - lo)  # (n_samples, dim)

    return physical.T  # (dim, n_samples) 관례 유지

## 1단계 — 데이터 로딩

In [ ]:
# TODO: PDD_bumper_SEA.ipynb에서 결정한 DESIGN_IDX, NOISE_IDX와 실제 (X, Y) 데이터로 교체할 것.
#
# 아직 DOE를 안 뽑았다면 lhs_design으로 새로 생성:
# domain_min = np.array([[1.5], [80], [60]])   # 예시: 두께/높이/폭 LB
# domain_max = np.array([[3.0], [120], [90]])  # 예시: UB
# X_physical = lhs_design(domain_min, domain_max, n_samples=150)  # 카티아/아바쿠스에 넘길 물리 단위
#
# 해석 다 돌리고 나면:
# X = X_physical  # (dim, N)
# Y = <아바쿠스 결과에서 뽑은 SEA>  # (N,)

X = None  # TODO
Y = None  # TODO
domain_min = None  # TODO
domain_max = None  # TODO

DESIGN_IDX = []  # TODO: PDD_bumper_SEA.ipynb 스크리닝 결과와 동일하게
NOISE_IDX = []   # TODO

## 2단계 — 크리깅 서로게이트 구축 및 검증

sklearn `GaussianProcessRegressor`는 입력을 (N, dim) 형태로 받음 — PDD 노트북의 (dim, N) 관례와 반대라 `.T` 필요.
학습 데이터만으로 R^2를 보지 않고 홀드아웃 검증셋으로 확인(PDD 노트북과 동일한 원칙).

In [ ]:
# X, Y, domain_min, domain_max가 채워지면 아래 실행

X_scaled = theoretical_scale(X, domain_min=domain_min, domain_max=domain_max)

X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled.T, Y, test_size=0.2, random_state=0
)

kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(1e-3)
gpr = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5, random_state=0)
gpr.fit(X_train, Y_train)

Y_pred = gpr.predict(X_test)
ss_res = np.sum((Y_test - Y_pred) ** 2)
ss_tot = np.sum((Y_test - np.mean(Y_test)) ** 2)
r2 = 1 - ss_res / ss_tot
rmse = np.sqrt(np.mean((Y_test - Y_pred) ** 2))

print(f"홀드아웃 검증: R^2={r2:.4f}, RMSE={rmse:.4f}")
print(f"학습된 커널: {gpr.kernel_}")

# TODO: R^2가 낮으면 커널 종류(nu 값, RBF 등) 바꿔보거나 샘플 추가.

## 3단계 — 강건 목적함수

`PDD_bumper_SEA.ipynb`와 같은 구조: 설계값(`DESIGN_IDX`)은 고정, 불확실 변수(`NOISE_IDX`)만 흔들어서 크리깅으로 SEA를 재평가 → mean/std 계산. 실제 FE를 다시 돌리는 게 아니라 이미 학습된 크리깅 모델만 재평가하는 거라 비용이 거의 없음.

In [ ]:
def estimate_robust_sea(d, model, design_idx=DESIGN_IDX, noise_idx=NOISE_IDX, n_mc=2000, seed=0):
    # d: design_idx에 대응하는 값들 (스케일링된 [-1,1] 공간 기준)
    dim = len(design_idx) + len(noise_idx)
    rng = np.random.default_rng(seed)

    x_full = np.zeros((n_mc, dim))
    x_full[:, design_idx] = np.array(d).reshape(1, -1)
    x_full[:, noise_idx] = rng.uniform(-1, 1, size=(n_mc, len(noise_idx)))

    sea_samples = model.predict(x_full)
    return sea_samples.mean(), sea_samples.std()


def robust_objective(d, kappa, model):
    mean_sea, std_sea = estimate_robust_sea(d, model)
    return -(mean_sea - kappa * std_sea)  # SEA 최대화 -> 최소화 문제로 부호 반전

## 4단계 — 최적화

In [ ]:
kappa = 1.0  # TODO: 강건성 가중치, 0~3 정도로 스윕하며 트레이드오프 확인

d0 = np.zeros(len(DESIGN_IDX))
bounds = [(-1, 1)] * len(DESIGN_IDX)

# TODO: 질량 상한, 최대 침입량/PCF 등 제약이 있으면 scipy 제약 형식으로 추가
result = minimize(robust_objective, d0, args=(kappa, gpr), method="SLSQP", bounds=bounds)

print(result)

## 5단계 — 검증

- 최적점(`result.x`)을 물리 단위로 되돌린 뒤(스케일링 역변환), 실제 카티아+아바쿠스로 재해석
- 크리깅 예측(mean_sea)과 실제 해석 결과 오차 확인
- 오차가 크면 그 근처에 DOE 점 추가 후 2단계부터 재적합